<a href="https://colab.research.google.com/github/dia-thoutam/infinilit/blob/main/Project2_BRCA_survival_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2 — Survival Prediction in Breast Cancer (TCGA-BRCA)

**Predicting patient survival from gene-expression and clinical data using statistical and machine-learning models.**

This notebook implements **every stage of the Project 2 workflow**:

| # | Workflow box | Section |
|---|--------------|---------|
| 1 | Download gene-expression + clinical data (TCGA-BRCA, UCSC Xena) | 2 |
| 2 | Preprocess (quality filter, log2, normalize, consistent IDs) | 3–4 |
| 3 | Data cleaning & integration (merge, drop unmatched) | 3 |
| 4 | Feature selection (top-variable + survival-associated genes) | 4, 6 |
| 5 | Survival modeling (Cox PH / Random Survival Forest **with cross-validation**) | 6–9 |
| 6 | Risk-group stratification (high vs. low risk from model scores) | 10 |
| 7 | Model evaluation (Concordance Index, Kaplan–Meier, log-rank) | 5, 8–10 |
| 8 | Biological interpretation (GO / pathway enrichment) | 11 |

**How to run:** In Google Colab, `Runtime → Run all` (or `Shift+Enter` cell by cell), top to bottom. No manual file upload — Section 2 downloads the data with `wget`.


## Section 1 — The concepts you need (Task 2)

### 1.1 Supervised vs. unsupervised learning
- **Supervised learning** uses data where each sample has a known *answer* (label); the model learns to map features to that answer. Predicting survival is supervised.
- **Unsupervised learning** uses *unlabelled* data to find structure — grouping similar patients (clustering) or compressing thousands of genes into a few axes (**PCA**). Used here for *exploration*, not prediction.

### 1.2 Classification vs. regression
- **Classification** predicts a *category* (e.g. "survived" vs. "died").
- **Regression** predicts a *number* (e.g. days to an event).
- **Survival analysis** is a special case: it predicts *time-to-event* while correctly handling **censoring**.

### 1.3 Censoring — why survival data is different
For most patients we never observe death; we only know they were alive at their last visit — a **censored** observation. Two columns encode this:
- `OS` — event indicator: `1` = died, `0` = censored (last known alive).
- `OS.time` — follow-up time in days.

A plain classifier would discard `OS.time` and mislabel censored patients, biasing the result. **Survival models use both columns**, so we use the survival form of every model below.

### 1.4 The models
- **Cox Proportional Hazard model (Cox PH)** — classic interpretable *statistical* baseline; estimates how each feature multiplies a patient's risk (hazard).
- **Random Survival Forest (RSF)** — many decision trees averaged; captures non-linear effects.
- **Gradient Boosting Survival** — trees built sequentially, each fixing the last one's errors (the *boosting* family; XGBoost's `cox` objective is a close cousin).
- **DeepSurv** — a neural network (deep learning) trained with the Cox partial-likelihood loss.

### 1.5 Scoring — the Concordance Index (C-index)
Every model outputs a **risk score**; we judge it by **Harrell's C-index** — the fraction of comparable patient pairs ranked correctly (higher risk → shorter survival). **0.5** = coin flip, **1.0** = perfect; **0.60–0.70** is typical and useful here.

### 1.6 Why cross-validation
A single train/test split can be lucky or unlucky. **K-fold cross-validation** rotates the test set through *K* slices of the data, so every patient is tested exactly once. Reporting the **mean ± standard deviation** of the C-index across folds is a far more trustworthy estimate of real performance (Section 9).


## Section 2 — Install packages and download the data (Tasks 5, box 1)

We install `lifelines` (Cox, Kaplan–Meier), `scikit-survival` (RSF, Gradient Boosting Survival), `pycox` (DeepSurv), and `gseapy` (gene-ontology / pathway enrichment). PyTorch is already in Colab.


In [ ]:
# ~2 minutes the first time
!pip install -q lifelines scikit-survival pycox torchtuples gseapy
print("Installation finished.")

In [ ]:
# Download data directly with wget -- no manual upload (Task 5 / box 1)
# Source: UCSC Xena, TCGA Breast Cancer (BRCA) cohort.

# Gene-expression matrix (IlluminaHiSeq RNA-seq), gzipped:
!wget -q -O HiSeqV2.gz "https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/HiSeqV2.gz"

# Clinical survival table:
!wget -q -O BRCA_survival.txt "https://tcga.xenahubs.net/download/survival/BRCA_survival.txt"

# Unzip -> file named "HiSeqV2"
!gunzip -f HiSeqV2.gz
print("Downloaded files:")
!ls -lh HiSeqV2 BRCA_survival.txt

## Section 3 — Load, clean and integrate the data (boxes 2–3)

The expression matrix arrives as *genes × samples*; we transpose it **exactly once** to *samples × genes*. (The earlier draft transposed twice, silently undoing the first — a subtle bug that is fixed here.)

**A note on the log2 transform (box 2):** UCSC Xena's `HiSeqV2` values are already `log2(normalized_count + 1)`, so the log2 step required by the workflow is *already applied by the data source*. We verify this by checking the value range, then normalise on top.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Clinical / survival data (rows = samples)
clinical_df = pd.read_csv("BRCA_survival.txt", sep="\t", index_col=0)
print("Clinical shape:", clinical_df.shape)
print(clinical_df[["OS", "OS.time"]].head())
print("\nMissing (OS, OS.time):\n", clinical_df[["OS", "OS.time"]].isnull().sum())
print("\nEvent distribution (0=censored/alive, 1=death):\n", clinical_df["OS"].value_counts())

In [ ]:
# Gene-expression data: arrives genes x samples -> transpose ONCE -> samples x genes
expression_df = pd.read_csv("HiSeqV2", sep="\t", index_col=0)
print("Raw expression (genes x samples):", expression_df.shape)
expression_df = expression_df.T
print("After single transpose (samples x genes):", expression_df.shape)

# Verify data is already log2-scaled (values should be small, roughly 0-20)
print("\nValue range -> min: %.2f  max: %.2f  (consistent with log2 counts)"
      % (expression_df.values.min(), expression_df.values.max()))

In [ ]:
# Integrate: keep only patients present in BOTH tables, drop missing outcomes (box 3)
clinical_df = clinical_df.dropna(subset=["OS", "OS.time"])
common = expression_df.index.intersection(clinical_df.index)
print("Matching samples:", len(common))

expression_df = expression_df.loc[common]
clinical_df   = clinical_df.loc[common]
print("Aligned expression:", expression_df.shape, "| clinical:", clinical_df.shape)

## Section 4 — Preprocessing & feature selection (boxes 2, 4)

1. Drop **zero-variance genes**.
2. Keep the **2,000 most variable genes** (feature selection, box 4).
3. **Standardise** each gene (mean 0, sd 1).
4. Remove **outlier samples** (IQR rule on mean expression — quality filter, box 2).
5. Compress to **50 principal components** — a shared, de-noised feature set that lets every model be compared fairly.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

gene_variance = expression_df.var(axis=0)
expression_df = expression_df.loc[:, gene_variance > 0]
print("Genes after zero-variance filter:", expression_df.shape[1])

top_genes = expression_df.var(axis=0).nlargest(2000).index
expression_df = expression_df[top_genes]
print("Genes after keeping top-2000 most variable:", expression_df.shape[1])

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(expression_df),
                        index=expression_df.index, columns=expression_df.columns)
print("Mean after scaling (~0): %.6f | Std (~1): %.6f"
      % (X_scaled.values.mean(), X_scaled.values.std()))

In [ ]:
# Outlier removal (IQR rule)
sm = X_scaled.mean(axis=1)
Q1, Q3 = sm.quantile(0.25), sm.quantile(0.75); IQR = Q3 - Q1
mask = (sm >= Q1 - 1.5*IQR) & (sm <= Q3 + 1.5*IQR)
X_scaled    = X_scaled[mask]
clinical_df = clinical_df.loc[X_scaled.index]
print("Outliers removed:", int((~mask).sum()), "| remaining:", int(mask.sum()))

In [ ]:
# PCA -> 50 shared components
N_PC = 50
pca = PCA(n_components=N_PC, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print("PCA feature matrix:", X_pca.shape)

plt.figure(figsize=(9,4))
plt.plot(range(1, N_PC+1), np.cumsum(pca.explained_variance_ratio_)*100, marker="o", ms=4)
plt.axhline(80, color="red", ls="--", label="80% threshold")
plt.xlabel("Number of principal components"); plt.ylabel("Cumulative variance (%)")
plt.title("PCA - cumulative variance explained"); plt.legend(); plt.tight_layout(); plt.show()
print("PC1: %.2f%% | PC2: %.2f%% of variance"
      % (pca.explained_variance_ratio_[0]*100, pca.explained_variance_ratio_[1]*100))

In [ ]:
# PC1 vs PC2 coloured by survival status
os_status = clinical_df["OS"].values
plt.figure(figsize=(8,6))
for s, c, lab in [(0,"steelblue","Censored (alive)"), (1,"crimson","Event (deceased)")]:
    m = os_status == s
    plt.scatter(X_pca[m,0], X_pca[m,1], c=c, label=lab, alpha=0.6, s=18)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("TCGA BRCA samples in PCA space, coloured by survival status")
plt.legend(); plt.tight_layout(); plt.show()

## Section 5 — Kaplan–Meier & log-rank, exploratory (box 7)

A first look: split patients by the **median of PC1** and compare survival curves with the **log-rank test** (p < 0.05 = significant difference). In Section 10 we repeat this using the *model's own risk score*, which is the version the workflow asks for.


In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

time  = clinical_df["OS.time"].values
event = clinical_df["OS"].values
high  = X_pca[:,0] > np.median(X_pca[:,0])

kmf = KaplanMeierFitter()
plt.figure(figsize=(8,6))
kmf.fit(time[high],  event[high],  label="High PC1");  kmf.plot_survival_function()
kmf.fit(time[~high], event[~high], label="Low PC1");   kmf.plot_survival_function()
plt.xlabel("Time (days)"); plt.ylabel("Survival probability")
plt.title("Kaplan-Meier curves split by PC1"); plt.tight_layout(); plt.show()

lr = logrank_test(time[high], time[~high], event[high], event[~high])
print("Log-rank p-value: %.4g | significant (p<0.05)? %s" % (lr.p_value, lr.p_value < 0.05))

## Section 6 — Cox model + survival-associated gene selection (boxes 4–5)

The **Cox model** is our statistical baseline (fitted on the first 10 PCs for a readable summary). We then run a **univariate Cox screen** across 300 genes to find those most associated with survival — this feeds both the feature-selection box and the biological interpretation in Section 11.


In [ ]:
from lifelines import CoxPHFitter

cox_df = pd.DataFrame(X_pca[:,:10], columns=[f"PC{i+1}" for i in range(10)], index=X_scaled.index)
cox_df["duration"] = clinical_df["OS.time"].values
cox_df["event"]    = clinical_df["OS"].values
cph = CoxPHFitter(penalizer=0.1).fit(cox_df, "duration", "event")
cph.print_summary()
print("Cox concordance index (in-sample):", round(cph.concordance_index_, 4))

In [ ]:
# Univariate Cox screen -> survival-associated genes (feature selection, box 4)
screen_genes = X_scaled.columns[:300]
records = []
for g in screen_genes:
    d = pd.DataFrame({"gene": X_scaled[g].values,
                      "duration": clinical_df["OS.time"].values,
                      "event": clinical_df["OS"].values})
    try:
        m = CoxPHFitter().fit(d, "duration", "event")
        records.append((g, m.summary.loc["gene","coef"], m.summary.loc["gene","p"]))
    except Exception:
        pass

screen = pd.DataFrame(records, columns=["gene","coef","p_value"]).sort_values("p_value")
sig_genes = screen[screen["p_value"] < 0.05]["gene"].tolist()
print("Genes screened:", len(screen), "| significant (p<0.05):", len(sig_genes))
print("\nTop 10 survival-associated genes:")
print(screen.head(10).to_string(index=False))

## Section 7 — Set up the shared data for the machine-learning models (box 5)

All models use the **same 50 PCA features** and the **same labels**, so the comparison is fair. We build the survival label array and a single held-out test split for a first quick look; Section 9 then does full cross-validation.


In [ ]:
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import train_test_split

X          = X_pca.astype("float32")                       # 50 PCA features
events_all = clinical_df["OS"].astype(bool).values         # True = death
times_all  = clinical_df["OS.time"].astype(float).values
y_all      = Surv.from_arrays(event=events_all, time=times_all)
pc_cols    = [f"PC{i+1}" for i in range(X.shape[1])]

def cindex(ev, tm, risk):
    return concordance_index_censored(ev.astype(bool), tm, risk)[0]

idx = np.arange(len(X))
idx_tr, idx_te = train_test_split(idx, test_size=0.25, random_state=42, stratify=events_all)
X_tr, X_te = X[idx_tr], X[idx_te]
y_tr, y_te = y_all[idx_tr], y_all[idx_te]
print("Train:", len(idx_tr), "| Test:", len(idx_te))

## Section 8 — Quick single-split fit of all four models (box 5)

In [ ]:
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from lifelines import CoxPHFitter
import torchtuples as tt
from pycox.models import CoxPH

single = {}

# Random Survival Forest
rsf = RandomSurvivalForest(n_estimators=200, min_samples_leaf=15, n_jobs=-1, random_state=42).fit(X_tr, y_tr)
single["Random Survival Forest"] = rsf.score(X_te, y_te)

# Gradient Boosting Survival (boosting)
gbs = GradientBoostingSurvivalAnalysis(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42).fit(X_tr, y_tr)
single["Gradient Boosting Survival"] = gbs.score(X_te, y_te)

# Cox PH on the same 50 PCs
cdf = pd.DataFrame(X_tr, columns=pc_cols); cdf["duration"]=times_all[idx_tr]; cdf["event"]=events_all[idx_tr]
cph_s = CoxPHFitter(penalizer=0.1).fit(cdf, "duration", "event")
risk_cox = cph_s.predict_partial_hazard(pd.DataFrame(X_te, columns=pc_cols)).values.flatten()
single["Cox PH"] = cindex(events_all[idx_te], times_all[idx_te], risk_cox)

# DeepSurv neural network
i_tr2, i_val = train_test_split(idx_tr, test_size=0.2, random_state=42, stratify=events_all[idx_tr])
net = tt.practical.MLPVanilla(X.shape[1], [32,32], 1, batch_norm=True, dropout=0.1, output_bias=False)
m = CoxPH(net, tt.optim.Adam); m.optimizer.set_lr(0.01)
m.fit(X[i_tr2], (times_all[i_tr2].astype("float32"), events_all[i_tr2].astype("float32")),
      batch_size=128, epochs=200, callbacks=[tt.callbacks.EarlyStopping()], verbose=False,
      val_data=(X[i_val], (times_all[i_val].astype("float32"), events_all[i_val].astype("float32"))))
single["DeepSurv (neural net)"] = cindex(events_all[idx_te], times_all[idx_te], m.predict(X_te).flatten())

print("Single-split test C-index:")
for k, v in sorted(single.items(), key=lambda kv: -kv[1]):
    print(f"  {k:<28} {v:.4f}")

## Section 9 — 5-fold cross-validation (rigorous comparison, box 5)

This is the version the workflow diagram asks for. We rotate a **5-fold** test set through the whole dataset, retrain all four models each time, and report the **mean ± standard deviation** C-index. This takes a few minutes (DeepSurv is retrained five times).


In [ ]:
from sklearn.model_selection import StratifiedKFold

K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)
cv = {"Cox PH": [], "Random Survival Forest": [], "Gradient Boosting Survival": [], "DeepSurv (neural net)": []}

for fold, (tr, te) in enumerate(skf.split(X, events_all), 1):
    ytr = Surv.from_arrays(event=events_all[tr], time=times_all[tr])
    yte = Surv.from_arrays(event=events_all[te], time=times_all[te])

    r = RandomSurvivalForest(n_estimators=200, min_samples_leaf=15, n_jobs=-1, random_state=42).fit(X[tr], ytr)
    cv["Random Survival Forest"].append(r.score(X[te], yte))

    g = GradientBoostingSurvivalAnalysis(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42).fit(X[tr], ytr)
    cv["Gradient Boosting Survival"].append(g.score(X[te], yte))

    cdf = pd.DataFrame(X[tr], columns=pc_cols); cdf["duration"]=times_all[tr]; cdf["event"]=events_all[tr]
    c = CoxPHFitter(penalizer=0.1).fit(cdf, "duration", "event")
    rk = c.predict_partial_hazard(pd.DataFrame(X[te], columns=pc_cols)).values.flatten()
    cv["Cox PH"].append(cindex(events_all[te], times_all[te], rk))

    t2, v = train_test_split(tr, test_size=0.2, random_state=42, stratify=events_all[tr])
    net = tt.practical.MLPVanilla(X.shape[1], [32,32], 1, batch_norm=True, dropout=0.1, output_bias=False)
    md_ = CoxPH(net, tt.optim.Adam); md_.optimizer.set_lr(0.01)
    md_.fit(X[t2], (times_all[t2].astype("float32"), events_all[t2].astype("float32")),
            batch_size=128, epochs=200, callbacks=[tt.callbacks.EarlyStopping()], verbose=False,
            val_data=(X[v], (times_all[v].astype("float32"), events_all[v].astype("float32"))))
    cv["DeepSurv (neural net)"].append(cindex(events_all[te], times_all[te], md_.predict(X[te]).flatten()))
    print(f"fold {fold}/{K} complete")

print("\n5-fold cross-validated C-index (mean +/- sd):")
cv_mean = {k: np.mean(v) for k, v in cv.items()}
for k, v in sorted(cv_mean.items(), key=lambda kv: -kv[1]):
    print(f"  {k:<28} {v:.4f} +/- {np.std(cv[k]):.4f}")
best_name = max(cv_mean, key=cv_mean.get)
print(f"\nBest model on this data: {best_name} (mean C-index {cv_mean[best_name]:.4f})")

In [ ]:
# Bar chart with error bars
order  = sorted(cv_mean, key=lambda k: -cv_mean[k])
means  = [cv_mean[k] for k in order]
sds    = [np.std(cv[k]) for k in order]
plt.figure(figsize=(8,5))
bars = plt.bar(order, means, yerr=sds, capsize=5,
               color=["crimson","steelblue","seagreen","goldenrod"])
plt.axhline(0.5, color="gray", ls="--", label="Random (0.5)")
plt.ylabel("Cross-validated C-index"); plt.title("Which model predicts BRCA survival best? (5-fold CV)")
plt.ylim(0.4, max(means)+0.1)
for b, m_ in zip(bars, means):
    plt.text(b.get_x()+b.get_width()/2, m_+0.01, f"{m_:.3f}", ha="center")
plt.xticks(rotation=20, ha="right"); plt.legend(); plt.tight_layout(); plt.show()

## Section 10 — Risk-group stratification (box 6)

We take the **best model** from cross-validation, refit it on all patients, compute each patient's **risk score**, and split the cohort at the **median** into **High-risk** and **Low-risk** groups. If the model is meaningful, the two Kaplan–Meier curves should separate clearly and the log-rank test should be significant.


In [ ]:
# Refit the best model on all data and produce a risk score per patient
if best_name == "Random Survival Forest":
    mdl = RandomSurvivalForest(n_estimators=200, min_samples_leaf=15, n_jobs=-1, random_state=42).fit(X, y_all)
    risk_all = mdl.predict(X)
elif best_name == "Gradient Boosting Survival":
    mdl = GradientBoostingSurvivalAnalysis(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42).fit(X, y_all)
    risk_all = mdl.predict(X)
elif best_name == "Cox PH":
    cdf = pd.DataFrame(X, columns=pc_cols); cdf["duration"]=times_all; cdf["event"]=events_all
    mdl = CoxPHFitter(penalizer=0.1).fit(cdf, "duration", "event")
    risk_all = mdl.predict_partial_hazard(pd.DataFrame(X, columns=pc_cols)).values.flatten()
else:  # DeepSurv
    t2, v = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42, stratify=events_all)
    net = tt.practical.MLPVanilla(X.shape[1], [32,32], 1, batch_norm=True, dropout=0.1, output_bias=False)
    mdl = CoxPH(net, tt.optim.Adam); mdl.optimizer.set_lr(0.01)
    mdl.fit(X[t2], (times_all[t2].astype("float32"), events_all[t2].astype("float32")),
            batch_size=128, epochs=200, callbacks=[tt.callbacks.EarlyStopping()], verbose=False,
            val_data=(X[v], (times_all[v].astype("float32"), events_all[v].astype("float32"))))
    risk_all = mdl.predict(X).flatten()

high_risk = risk_all > np.median(risk_all)
print(f"Stratifying with the best model: {best_name}")
print("High-risk patients:", int(high_risk.sum()), "| Low-risk patients:", int((~high_risk).sum()))

In [ ]:
# Kaplan-Meier + log-rank on the model-defined risk groups
kmf = KaplanMeierFitter()
plt.figure(figsize=(8,6))
kmf.fit(times_all[high_risk],  events_all[high_risk],  label="High risk"); kmf.plot_survival_function()
kmf.fit(times_all[~high_risk], events_all[~high_risk], label="Low risk");  kmf.plot_survival_function()
plt.xlabel("Time (days)"); plt.ylabel("Survival probability")
plt.title(f"Risk-group survival ({best_name})"); plt.tight_layout(); plt.show()

lr = logrank_test(times_all[high_risk], times_all[~high_risk],
                  events_all[high_risk], events_all[~high_risk])
print("Log-rank p-value: %.4g | significant separation (p<0.05)? %s"
      % (lr.p_value, lr.p_value < 0.05))

## Section 11 — Biological interpretation (box 8)

Finally we ask *what the top predictor genes actually do*. We take the **survival-associated genes** from the univariate screen (Section 6) and run **gene-ontology (GO) and KEGG pathway enrichment** with `gseapy`'s Enrichr interface. Enriched terms suggest which biological processes drive the survival signal — the material for your discussion section.

*(This step queries the online Enrichr service, so it needs an internet connection, which Colab has.)*


In [ ]:
import gseapy as gp

# Use significant survival genes if we have enough, else the top of the screen
gene_list = sig_genes if len(sig_genes) >= 15 else screen.head(100)["gene"].tolist()
print("Genes sent to enrichment:", len(gene_list))

try:
    enr = gp.enrichr(gene_list=gene_list,
                     gene_sets=["GO_Biological_Process_2021", "KEGG_2021_Human"],
                     organism="human", outdir=None)
    res = enr.results.sort_values("Adjusted P-value").head(15)
    print(res[["Gene_set", "Term", "Adjusted P-value", "Genes"]].to_string(index=False))
except Exception as e:
    res = None
    print("Enrichment call failed (network?):", e)

In [ ]:
# Plot the top enriched terms
if res is not None and len(res) > 0:
    top = res.head(10).iloc[::-1]
    plt.figure(figsize=(9,6))
    plt.barh(top["Term"].str.slice(0,45), -np.log10(top["Adjusted P-value"]), color="teal")
    plt.xlabel("-log10(adjusted p-value)")
    plt.title("Top enriched GO / KEGG terms among survival-associated genes")
    plt.tight_layout(); plt.show()
else:
    print("No enrichment results to plot.")

## Section 12 — Reading your results & writing up

**Which model is best?** The winner is the one with the highest **cross-validated** mean C-index (Section 9) — trust that number over the single split in Section 8. Values of 0.60–0.70 are normal and genuinely useful; do not expect near-1.0. Because folds are random, re-running with a different `random_state` shifts the numbers slightly; the mean ± sd already communicates that uncertainty.

**Points to discuss in the report:**
- *Why survival models, not classifiers?* Most patients are censored; ignoring `OS.time` would waste the majority of the information and bias the result (Section 1.3).
- *Cox vs. machine learning.* Cox assumes each feature has a constant, linear effect on log-risk. RSF, boosting and DeepSurv can model curves and interactions — an advantage on complex signals, but they can also overfit a dataset this size, which is exactly why cross-validation matters.
- *Risk stratification (Section 10).* A significant log-rank p-value between the model's high- and low-risk groups is direct evidence the risk score is clinically meaningful.
- *Biological interpretation (Section 11).* Enriched GO/KEGG terms connect the statistics back to biology (e.g. cell-cycle, immune, or metabolic pathways) — the "so what" of the project.
- *Limitations.* Single cohort; expression-only features (no clinical stage/age); enrichment depends on the gene screen size. Adding clinical covariates and an external validation cohort (e.g. METABRIC) would strengthen the conclusions.

This notebook now covers **all eight boxes** of the Project 2 workflow end to end.
